# Sprint 4 · Semana 11: mejora del modelo con otro algoritmo y otra arquitectura

Tesis: Detección Automatizada y Priorización de Eventos Adversos en Notas Clínicas mediante un Flujo de Procesamiento (Pipeline) de NLP y Aprendizaje Automático — Aplicación de la Matriz de Priorización del Modelo GEMSES sobre MIMIC-IV

Autor: Carlos Pérez Pérez · Curso: MIA-14 Trabajo de Investigación II · Docente: Glen Rodríguez · 26 de noviembre de 2026

Este cuaderno reúne el entregable de la semana 11 y se entrega en la rama `sprint4-semana11-mejora-modelo`. El enunciado pide mejorar
el modelo con mejores datos, más ingeniería de atributos o más optimización de hiperparámetros. Si los modelos no mejoran, permite probar
otro algoritmo u otra arquitectura.

El cuaderno no entrena modelos. Lee los resultados que deja el *script* `src/experimentos_semana11.py` en `reportes/` y en `logs/`. Las
cifras de esta semana se pueden verificar en las celdas de código sin acceso a los datos de MIMIC-IV, que protege el acuerdo de uso de
PhysioNet. Las cifras de semanas anteriores llevan su fuente.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Image, display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
leer = lambda p: json.loads((RAIZ / p).read_text(encoding="utf-8"))
R, S8 = leer("reportes/semana11_resultados.json"), leer("reportes/hp_pruning_resultados.json")
tab = pd.read_csv(RAIZ / "reportes" / "semana11_folds.csv")
curva = pd.read_csv(RAIZ / "reportes" / "hp_pruning_curva.csv")
CFG = yaml.safe_load((RAIZ / "config" / "config.yaml").read_text(encoding="utf-8"))
NOMBRES = {"referencia": "Referencia: logística A+B+C", "E1_nblr": "E1: NB-LR", "E2_fusion_tardia": "E2: fusión tardía"}
print("Generado:", R["generado"], "· scikit-learn", R["sklearn"], "· semilla", R["semilla"])
print("Protocolo:", R["protocolo"])

Generado: 2026-11-26T19:33:01 · scikit-learn 1.7.2 · semilla 42
Protocolo: StratifiedGroupKFold(5, shuffle=True, random_state=42) por subject_id en train (los folds de las semanas 5 a 8); entrenan las filas de D2; métrica: PR-AUC en las epicrisis de validación sin etiqueta dudosa; test sin medir


## 1. Por qué otro algoritmo y otra arquitectura

El enunciado ofrece tres caminos para mejorar el modelo. Si los modelos no mejoran, permite un cuarto, que es probar otro algoritmo u
otra arquitectura. Esta semana se tomó ese cuarto camino, por lo que mostraron las semanas anteriores en los otros tres.

La optimización de hiperparámetros ya se hizo en las semanas 7 y 8. En la semana 8, la mejor configuración superó a la actual en
5 de 5 *folds*, con una diferencia media de PR-AUC (área bajo la curva de precisión y
sensibilidad) de 0.0014. En *test*, su PR-AUC fue 0.8670, frente a
0.8671 de la configuración actual. Por ello, una búsqueda más en el mismo espacio tiene poco recorrido.

Los atributos también se probaron. La ablación de la semana 5 comparó las palabras (A), los pares de palabras (B) y las variables de la
hospitalización con el largo de la nota (C), y se adoptó A+B+C. Esta semana se usan los mismos bloques y el mismo vocabulario. E1 cambia
la forma de ponderar los términos de A y B, y E2 cambia la forma de combinar los bloques.

Los datos también se probaron. En la semana 6 se compararon tres versiones de las filas de entrenamiento (D0, D1 y D2). D2, que quita del
entrenamiento las negativas y las positivas con etiqueta dudosa, subió la PR-AUC de validación cruzada de 0.8431 a 0.8534, sobre las
mismas filas. En la semana 7, la variante D3 no reemplazó a D2. Ambos resultados están en el README, en las secciones de las semanas 6 y 7.

Más datos sí podrían ayudar. La curva de aprendizaje de la semana 8 se calculó con la configuración ganadora de esa semana. En ella, la
PR-AUC de validación subió 0.0105 en promedio al pasar de unas 19 599 a unas 39 198 filas de
entrenamiento, y subió en 5 de 5 *folds*. Sin embargo, el corpus se limita a 70 000 epicrisis
por la memoria del equipo (`n_max` en `config/config.yaml`). Además, las epicrisis nuevas deben quedar fuera de la partición de prueba.
También deben quedar fuera de las 98 931 epicrisis de pacientes que no figuran en el corpus, de las que sale la revisión ciega (Enmienda 8,
sección 1, numeral 4). Así, esas epicrisis siguen siendo de pacientes que no participaron en el desarrollo del modelo. Ese trabajo queda
para un *sprint* posterior.

In [2]:
c = curva.pivot(index="fold", columns="fraccion", values="pr_auc_validacion")
f = curva.pivot(index="fold", columns="fraccion", values="filas_entrenamiento")
g = S8["regla"]
print("Semana 8 · la mejor configuración superó a la actual en", g["folds_mejores"], "de", g["n_folds"],
      "folds · diferencia media:", round(g["dif_media"], 4))
print("Semana 8 · test, actual y candidata:", S8["test"]["actual"]["comun"]["pr_auc"], S8["test"]["candidata"]["comun"]["pr_auc"])
print("Semana 8 · curva de aprendizaje de la ganadora · filas de entrenamiento (media) con las fracciones 0.5 y 1:",
      round(f[0.5].mean()), "y", round(f[1.0].mean()))
print("Semana 8 · cambio de la PR-AUC de validación entre 0.5 y 1, por fold:", (c[1.0] - c[0.5]).round(4).tolist(),
      "· media:", round((c[1.0] - c[0.5]).mean(), 4))
print("Techo del corpus por memoria (n_max en config/config.yaml):", CFG["preprocesado"]["n_max"])

Semana 8 · la mejor configuración superó a la actual en 5 de 5 folds · diferencia media: 0.0014
Semana 8 · test, actual y candidata: 0.8671 0.867
Semana 8 · curva de aprendizaje de la ganadora · filas de entrenamiento (media) con las fracciones 0.5 y 1: 19599 y 39198
Semana 8 · cambio de la PR-AUC de validación entre 0.5 y 1, por fold: [0.0106, 0.009, 0.0109, 0.0114, 0.0104] · media: 0.0105
Techo del corpus por memoria (n_max en config/config.yaml): 70000


## 2. Diseño, fijado antes de correr

El diseño está en `config/semana11_mejora.yaml` y se registró en el *commit* 2aa27ee, antes de la corrida completa. La prueba del flujo,
con un *fold* y una muestra de 4 000 filas de entrenamiento y 2 000 de validación, corrió 39 segundos antes de ese *commit*. Su *log*
también se publica y lleva los mismos parámetros que el de la corrida completa. El diseño va en un archivo aparte para no tocar
`config/config.yaml`, que lleva el prerregistro de la tesis.

La referencia es la configuración actual de la tesis. Se trata de una regresión logística con los bloques A, B y C, entrenada con las
filas de la variante D2. A y B usan 20 000 términos cada uno, con frecuencia mínima de 3 documentos y TF-IDF sublineal. La regresión usa
C = 1.0 y pesos balanceados. Es la configuración del lote de la revisión ciega y la que usa la HE1.

La referencia no es la ganadora de la semana 8, aunque la regla la adoptó. En la semana 8 se decidió evaluar la HE1 con la configuración
actual y repetirla con la ganadora como análisis de sensibilidad (README, semana 8). Por ello, la comparación principal es con la
configuración actual, y en la sección 4 también se compara con la ganadora.

El experimento E1 prueba otro algoritmo, NB-LR, que combina Naive Bayes con la regresión logística. Wang y Manning (2012) proponen usar
como atributo la presencia de cada término multiplicada por la razón de log-conteos de Naive Bayes. Sobre esos atributos entrenan una
SVM lineal (p. 91). La razón se define como r = log((p / ‖p‖₁) / (q / ‖q‖₁)). En ella, p y q suman la presencia de cada término en los
casos con evento y sin evento, más un suavizado α (pp. 90-91). Los autores usan α = 1 y C = 1 en NBSVM (p. 92). También señalan que la
regresión logística en lugar de la SVM da resultados parecidos (p. 93).

En este experimento, r se calcula solo con las filas de entrenamiento de cada *fold*. El bloque C entra sin cambios, y el clasificador es
la misma regresión de la referencia, con C = 1.0. No se usa la interpolación β de NBSVM.

El experimento E2 prueba otra arquitectura, la fusión tardía. La referencia junta todos los bloques en un solo modelo lineal. E2 lo
separa en dos etapas. En la primera, la regresión logística con solo el texto (A+B) da un puntaje. En la segunda, un
*HistGradientBoostingClassifier* (scikit-learn, s. f.) combina ese puntaje con el bloque C y puede captar relaciones no lineales del
contexto. Para que la segunda etapa no aprenda de un puntaje sobreajustado, el puntaje de entrenamiento se calcula fuera de *fold*, con
3 *folds* internos por paciente. En validación, la segunda etapa recibe el puntaje de la regresión ajustada con todas las filas de
entrenamiento del *fold*. Ese modelo usa más filas que los de los *folds* internos, de modo que su escala puede diferir un poco. Los
valores del *boosting* son fijos y no se buscaron.

La evaluación es la de las semanas 5 a 8. Se usan los 5 *folds* del proyecto, separados por paciente con la semilla 42, solo dentro de
*train*. La PR-AUC se mide en las epicrisis de validación sin etiqueta dudosa. La regla es la prerregistrada en la semana 7. Un
experimento mejora si supera a la referencia en al menos 4 de los 5 *folds*. Además, su mejora media debe ser mayor que una desviación
estándar de las diferencias por *fold*.

El diseño fija que el *test* no se mide esta semana (`test: no`), y sus filas se descartan al cargar los datos, antes de cualquier
cálculo. La HE1 se calcula en la partición de prueba, y su contraste, registrado en la Enmienda 8 (*commit* dc43b22), todavía no se
ejecuta. Según el mismo diseño, el *test* de un experimento que cumpla la regla se mide después de ese contraste.

In [3]:
D = yaml.safe_load((RAIZ / "config" / "semana11_mejora.yaml").read_text(encoding="utf-8"))["semana11_mejora"]
print("Referencia:", R["referencia"])
print("E1, alfa:", D["experimentos"]["E1_nblr"]["alfa"], "· E2, folds internos:", D["experimentos"]["E2_fusion_tardia"]["folds_internos"])
print("E2, HistGradientBoosting:", D["experimentos"]["E2_fusion_tardia"]["hgb"])
print("Regla:", D["regla"], "· igual a la de config.yaml:", D["regla"] == CFG["hiperparametros"]["regla"])
print("Test:", D["test"], "· filas de test descartadas sin usarse:", R["filas_test_descartadas"])
for p in sorted((RAIZ / "logs").glob("experimentos_semana11*.log")):
    print("Inicio del log", p.name, "·", p.read_text(encoding="utf-8").split(" | ")[0])

Referencia: {'C': 1.0, 'A_max_features': 20000, 'A_min_df': 3, 'B_max_features': 20000, 'B_min_df': 3, 'class_weight': 'balanced', 'sublinear_tf': True}
E1, alfa: 1.0 · E2, folds internos: 3
E2, HistGradientBoosting: {'learning_rate': 0.05, 'max_iter': 300, 'max_leaf_nodes': 31, 'min_samples_leaf': 50, 'l2_regularization': 1.0, 'early_stopping': False, 'class_weight': 'balanced', 'random_state': 42}
Regla: {'folds_mejores_minimo': 4, 'desviaciones_minimas': 1.0} · igual a la de config.yaml: True
Test: False · filas de test descartadas sin usarse: 14085
Inicio del log experimentos_semana11_20261126_190948.log · 2026-11-26 19:09:48
Inicio del log experimentos_semana11_prueba_20261126_190901.log · 2026-11-26 19:09:01


## 3. Control de reproducción

Antes de comparar, se comprobó que la referencia parte del mismo punto que la semana 8. Como se aprecia en la Tabla 1, en cada uno de los
5 *folds* su PR-AUC coincide con la confirmación de esa semana, con una diferencia máxima de 0. En consecuencia, las diferencias que
siguen se deben al algoritmo o a la arquitectura, y no a un cambio en los datos o en la partición.

Tabla 1. PR-AUC de la referencia y de la confirmación de la semana 8, por *fold*

In [4]:
ref = tab[tab.experimento == "referencia"][["fold", "pr_auc_comun", "control_semana8", "control_dif"]]
display(ref.rename(columns={"fold": "Fold", "pr_auc_comun": "Referencia, semana 11", "control_semana8": "Confirmación, semana 8",
                            "control_dif": "Diferencia"}).set_index("Fold").round(6))
print("Diferencia máxima con la semana 8:", ref.control_dif.abs().max())

,"Referencia, semana 11","Confirmación, semana 8",Diferencia
Fold,,,
1,0.856697,0.856697,0.0
2,0.852738,0.852738,0.0
3,0.846717,0.846717,0.0
4,0.854189,0.854189,0.0
5,0.856627,0.856627,0.0


Diferencia máxima con la semana 8: 0.0


Fuente: elaboración propia a partir de [`reportes/semana11_folds.csv`](../reportes/semana11_folds.csv) y
[`reportes/hp_pruning_resultados.json`](../reportes/hp_pruning_resultados.json).

## 4. Resultados

Como se aprecia en la Tabla 2, ninguno de los dos experimentos superó a la referencia. La referencia obtuvo una PR-AUC media de
0.8534. E2 obtuvo 0.8525, con una diferencia media de -0.0009, y quedó por debajo en
5 de 5 *folds*. E1 obtuvo 0.8044, con una diferencia media de
-0.0490, y quedó por debajo en 5 de 5 *folds*.

Tabla 2. PR-AUC por *fold* en las epicrisis de validación sin etiqueta dudosa

In [5]:
t1 = tab.pivot(index="fold", columns="experimento", values="pr_auc_comun")[list(NOMBRES)].rename(columns=NOMBRES)
t1.loc["Media"] = t1.mean()
t1.loc["Desviación estándar"] = t1.iloc[:5].std(ddof=1)
display(t1.round(4))

experimento,Referencia: logística A+B+C,E1: NB-LR,E2: fusión tardía
fold,,,
1,0.8567,0.8081,0.8547
2,0.8527,0.8050,0.8525
3,0.8467,0.7979,0.8467
4,0.8542,0.8030,0.8526
5,0.8566,0.8078,0.8560
Media,0.8534,0.8044,0.8525
Desviación estándar,0.0041,0.0042,0.0036


Fuente: elaboración propia a partir de [`reportes/semana11_folds.csv`](../reportes/semana11_folds.csv).

En la Figura 1 se observa que las líneas de la referencia y de E2 casi se superponen en los 5 *folds*. E1 queda unos 0.05 por debajo
en todos.

Figura 1. PR-AUC por *fold* de la referencia y de los dos experimentos

![PR-AUC por fold](../reportes/figuras/fig_semana11_folds.png)

Fuente: elaboración propia a partir de [`reportes/semana11_folds.csv`](../reportes/semana11_folds.csv).

En la Tabla 3 se resume la regla. La ROC-AUC, que es la métrica de la HE1, sigue el mismo orden. La referencia obtuvo 0.8582,
E2 obtuvo 0.8575 y E1 obtuvo 0.8037.

Tabla 3. Regla de decisión frente a la referencia

In [6]:
filas = []
for e in ("E1_nblr", "E2_fusion_tardia"):
    g = R["regla"][e]
    filas.append({"Experimento": NOMBRES[e], "PR-AUC media": R["resumen"][e]["pr_auc_comun"]["media"],
                  "Diferencia media": g["dif_media"], "Desviación de las diferencias": g["dif_std"],
                  "Folds mejores": f'{g["folds_mejores"]} de {g["n_folds"]}', "ROC-AUC media": R["resumen"][e]["roc_auc_comun"]["media"],
                  "Cumple la regla": "sí" if g["adopta_candidata"] else "no"})
filas.insert(0, {"Experimento": NOMBRES["referencia"], "PR-AUC media": R["resumen"]["referencia"]["pr_auc_comun"]["media"],
                 "ROC-AUC media": R["resumen"]["referencia"]["roc_auc_comun"]["media"]})
display(pd.DataFrame(filas).set_index("Experimento").round(4).fillna(""))
print("Decisión:", R["decision"])

,PR-AUC media,ROC-AUC media,Diferencia media,Desviación de las diferencias,Folds mejores,Cumple la regla
Experimento,,,,,,
Referencia: logística A+B+C,0.8534,0.8582,,,,
E1: NB-LR,0.8044,0.8037,-0.049,0.0013,0 de 5,no
E2: fusión tardía,0.8525,0.8575,-0.0009,0.0008,0 de 5,no


Decisión: ningún experimento cumple la regla; se mantiene la referencia


Fuente: elaboración propia a partir de [`reportes/semana11_resultados.json`](../reportes/semana11_resultados.json).

Frente a la ganadora de la semana 8, la distancia es mayor. Como se aprecia en la Tabla 4, la diferencia media de E2 con la ganadora es
-0.0023 y la de E1 es -0.0504. Los dos quedan por debajo en los
5 *folds*. La de la referencia es -0.0014, que es el resultado de la confirmación de
la semana 8 visto desde la ganadora.

Tabla 4. Diferencia de PR-AUC con la ganadora de la semana 8, con los mismos 5 *folds*

In [7]:
gan = pd.Series({int(f["particion"].rsplit("_f", 1)[1]): f["pr_auc_comun"] for f in S8["confirmacion_5_folds"]["candidata"]["folds"]})
filas = []
for e in NOMBRES:
    x = tab[tab.experimento == e].set_index("fold").pr_auc_comun
    dif = x - gan
    filas.append({"Modelo": NOMBRES[e], "PR-AUC media": x.mean(), "Diferencia media con la ganadora": dif.mean(),
                  "Desviación de las diferencias": dif.std(ddof=1), "Folds por debajo": f"{(dif < 0).sum()} de {len(dif)}"})
display(pd.DataFrame(filas).set_index("Modelo").round(4))
print("Ganadora de la semana 8:", S8["ganadora"], "· PR-AUC media en 5 folds:", round(gan.mean(), 4))

,PR-AUC media,Diferencia media con la ganadora,Desviación de las diferencias,Folds por debajo
Modelo,,,,
Referencia: logística A+B+C,0.8534,-0.0014,0.0006,5 de 5
E1: NB-LR,0.8044,-0.0504,0.0011,5 de 5
E2: fusión tardía,0.8525,-0.0023,0.0010,5 de 5


Ganadora de la semana 8: {'C': 1.8734194377493316, 'A_max_features': 60000, 'A_min_df': 3, 'B_max_features': 40000, 'B_min_df': 5, 'class_weight': 'ninguno', 'sublinear_tf': True} · PR-AUC media en 5 folds: 0.8547


Fuente: elaboración propia a partir de [`reportes/semana11_folds.csv`](../reportes/semana11_folds.csv) y
[`reportes/hp_pruning_resultados.json`](../reportes/hp_pruning_resultados.json).

## 5. Lectura de los resultados

E2 quedó muy cerca de la referencia, aunque por debajo en todos los *folds*. Su primera etapa, solo con el texto, obtuvo una PR-AUC media
de 0.8466. La segunda etapa, con el contexto, la llevó a 0.8525, sin superar a la referencia. Es decir, el *boosting* sobre las
variables de la hospitalización, con los valores fijos que se usaron, no aportó más que el modelo lineal con todos los bloques juntos.
Además, E2 tiene dos etapas y es más difícil de explicar.

E1 quedó por debajo de la referencia en todos los *folds*. Wang y Manning (2012) evaluaron NBSVM en tareas de sentimiento, subjetividad y
tópicos (p. 93), no en notas clínicas. El resultado vale para NB-LR con C = 1.0 y sin normalizar, y no descarta otras versiones. Una
explicación posible, que no se probó, es la escala de los atributos. La presencia multiplicada por r no se normaliza por el largo de la
nota, a diferencia del TF-IDF de la referencia. Además, la regresión se usó con el mismo C = 1.0 sin ajustarlo, y C fue el hiperparámetro
de mayor importancia en la semana 8 (0.8337 con PED-ANOVA).

Una vez analizados los resultados, se concluye que ninguno de los dos experimentos cumple la regla, ni frente a la configuración actual
ni frente a la ganadora de la semana 8. Por ello, no cambia ninguna configuración. La HE1 sigue con la configuración actual, y con la
ganadora como análisis de sensibilidad.

Tiempo de cálculo en los 5 *folds*: 7.7 minutos la referencia, 10.4 E1 y 4.9
E2. La corrida completa tomó 23.2 minutos, y el resto corresponde a la carga y la preparación de los datos.

In [8]:
print("PR-AUC media de la primera etapa de E2, solo con el texto:",
      round(tab[tab.experimento == "E2_fusion_tardia"].pr_auc_comun_etapa1_solo_texto.mean(), 4))
print("Minutos por experimento:", (tab.groupby("experimento").segundos.sum() / 60).round(1).to_dict(),
      "· corrida completa:", round(R["segundos_corrida"] / 60, 1))
print("Importancia de los hiperparámetros en la semana 8:", S8["importancia"]["importancias"])

PR-AUC media de la primera etapa de E2, solo con el texto: 0.8466
Minutos por experimento: {'E1_nblr': 10.4, 'E2_fusion_tardia': 4.9, 'referencia': 7.7} · corrida completa: 23.2
Importancia de los hiperparámetros en la semana 8: {'C': 0.8337, 'B_max_features': 0.0918, 'B_min_df': 0.0322, 'A_max_features': 0.0239, 'A_min_df': 0.0147, 'sublinear_tf': 0.0037, 'class_weight': 0.0}


## 6. Relación con la tesis

Según la Enmienda 7, sección 2.5 (*commit* 93b96eb), ninguna mejora reemplaza un análisis registrado. Una mejora se informa aparte,
rotulada como análisis no prerregistrado, y se compara con el análisis registrado. Estos experimentos siguen esa regla. No cambian la
configuración con la que se sortearon las epicrisis de la revisión ciega ni la de la HE1, y no usan la partición de prueba.

## 7. Siguiente paso

Se recomienda probar E1 con la presencia normalizada por fila y con C buscado en *folds* internos dentro de *train*, como variante nueva
registrada antes de correr. La interpolación β no es prioridad. Wang y Manning (2012) informan que, en documentos largos, NBSVM rinde casi
igual con β entre ¼ y 1, con diferencias de 0.1 % como máximo (p. 93).

También se recomienda ampliar las epicrisis de entrenamiento, porque la curva de aprendizaje de la semana 8 todavía sube. Para ello hay
que subir el techo de memoria del corpus y elegir epicrisis fuera de la partición de prueba y fuera de las 98 931 de la revisión ciega.

Por último, queda por probar un modelo neuronal, como un *transformer* preentrenado con textos clínicos, también registrado antes de
correr.

## Cómo reproducir la semana 11

1) Requisitos: los de la semana 8 (el `.env`, `data/processed/dataset.parquet`, las cachés de dominio y de variantes) y la caché de
conteos de los 5 *folds* de confirmación en `data/interim/cache_hp/`, que arma `python src/experimentos_hp.py --solo-conteos`.

2) Comando único: `python src/experimentos_semana11.py`. Escribe `reportes/semana11_folds.csv`, `reportes/semana11_resultados.json`,
la figura y su *log* en `logs/`. Con `--prueba` corre un *fold* con una muestra de filas y escribe en `data/interim/prueba_semana11/`.

3) Si la corrida se interrumpe, al repetir el comando se toman de `data/interim/semana11_folds.json` los *folds* ya evaluados.

4) Los resultados de este cuaderno se obtuvieron con el *script* del *commit* 2aa27ee.

## Referencias

scikit-learn (s. f.). *sklearn.ensemble.HistGradientBoostingClassifier*. Documentación de scikit-learn, versión 1.7.2 (*docstring* de
la librería instalada).

Wang, S. y Manning, C. D. (2012). Baselines and bigrams: Simple, good sentiment and topic classification. En *Proceedings of the 50th
Annual Meeting of the Association for Computational Linguistics (Volume 2: Short Papers)* (pp. 90-94). Association for Computational
Linguistics. https://aclanthology.org/P12-2018/